# 01. PDF 직접 로딩 기반 Naive RAG의 한계

## Naive RAG의 한계

| 문제 | 원인 | 영향 |
|---|---|---|
| 검색 품질 불안정 | 단일 쿼리에만 의존 | 질문 표현에 따라 결과 편차 |
| 노이즈 컨텍스트 | 관련 없는 청크도 k개 무조건 반환 | LLM 답변 품질 저하 |
| 컨텍스트 중복 | 유사한 청크가 여러 개 검색됨 | 토큰 낭비 |
| 할루시네이션 | 검색 실패 시 LLM 자체 지식으로 답변 | 부정확한 답변 |

## Advanced RAG 개선 방향
```
Naive RAG:    질문 → 검색 → 생성
Advanced RAG: 
- PDF를 Markdown으로 변환하여 문서 구조를 보존한 뒤 로딩
- 질문 → [쿼리 재작성 / 다중쿼리] → 검색 → [재순위화 / 압축] → 생성
```

## 학습 목표

이번 실습에서는 PDF 문서를 직접 로딩하여 기본 RAG 구조를 구현하고, Naive RAG가 가지는 한계를 확인한다.
실습 후 다음 내용을 설명할 수 있다.

1. PDF 문서를 `PyPDFLoader`로 로딩하는 방법
2. 문서를 청크 단위로 분할하고 ChromaDB에 저장하는 방법
3. 단일 쿼리 기반 검색이 질문 표현에 따라 달라지는 이유
4. 고정된 `k`개 문서 검색 방식의 한계
5. PDF 직접 로딩 방식이 표, 목차, 페이지 구조에서 약해질 수 있는 이유
6. Advanced RAG가 필요한 이유

## 실습 데이터
**「2026 AI 주요 트렌드 전망」NIA AI 트렌드 보고서 기반 한국어 RAG 챗봇 만들기** 

# 환경변수 로딩

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

True

# ChromaDB로 Vector DB 구성

ChromaDB는 로컬 파일에 영구 저장(persist)이 가능한 오픈소스 벡터 DB이다.

In [2]:
# 추가 설치 필요 패키지
# uv add chromadb langchain-chroma pypdf langchain-community langchain-text-splitters langchain-openai 
# uv add chromadb langchain-chroma pypdf

## PDF 파일 로딩

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


PDF_PATH = Path("data/AI@Data_Report_2026_전망_251223(최종).pdf")

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {PDF_PATH}")


# PDF 로딩 (페이지 단위 Document 생성)
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()
print(f"PDF 페이지 수: {len(pages)}")
print(f"1페이지 미리보기:\n{pages[0].page_content[:300]}")

C:\Users\magpi\AppData\Local\Temp\ipykernel_29952\263116575.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


PDF 페이지 수: 22
1페이지 미리보기:
AI@Data Report
                                     2025-제3호
목차❶ 배경 및 목적 /p.06❷ 자료 분석 및 방법 /p.09❸ 분석 결과 /p.12❹ 2026년 전망: AI 생태계 방향성 /p.17❺ 분석 결과로 본 우리의 대응 및 노력 /p.20
토픽 분석을 통한AI 주요 트렌드 및 2026 전망


## 청크 분할

In [4]:
# 청크 분할
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n\n", "\n", ".", " ", ""]
)
chunks = splitter.split_documents(pages)

# 청크 번호를 metadata에 추가한다.
for i, chunk in enumerate(chunks, start=1):
    chunk.metadata["chunk_id"] = i

print(f"분할된 청크 수: {len(chunks)}")
print(f"청크 크기: 500")
print(f"청크 중복: 80")

print("\n[청크 예시]")
print("metadata:", chunks[0].metadata)
print(chunks[0].page_content[:300])


분할된 청크 수: 52
청크 크기: 500
청크 중복: 80

[청크 예시]
metadata: {'producer': 'Hancom PDF 1.3.0.550', 'creator': 'Hwp 2020 11.0.0.8808', 'creationdate': '2025-12-23T10:45:53+09:00', 'author': 'choro', 'moddate': '2025-12-23T10:45:53+09:00', 'pdfversion': '1.4', 'source': 'data\\AI@Data_Report_2026_전망_251223(최종).pdf', 'total_pages': 22, 'page': 0, 'page_label': '1', 'chunk_id': 1}
AI@Data Report
                                     2025-제3호
목차❶ 배경 및 목적 /p.06❷ 자료 분석 및 방법 /p.09❸ 분석 결과 /p.12❹ 2026년 전망: AI 생태계 방향성 /p.17❺ 분석 결과로 본 우리의 대응 및 노력 /p.20
토픽 분석을 통한AI 주요 트렌드 및 2026 전망


## ChromaDB 생성 또는 로딩

In [ ]:
from pathlib import Path

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# 임베딩 모델 초기화
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# ChromaDB 경로 및 컬렉션명
DB_PATH = Path("chroma_db/ai_trend_report_pdf")
COLLECTION_NAME = "ai_trend_report_2026_pdf"

# 기존 DB가 있으면 로딩
if DB_PATH.exists():
    print("기존 ChromaDB가 존재하므로 새로 생성하지 않고 로딩합니다.")
    # ChromaDB 기존 컬렉션 로딩
    vector_store = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=str(DB_PATH)
    )

# 기존 DB가 없으면 새로 생성
else:
    print("기존 ChromaDB가 없어 새로 생성합니다.")
    # ChromaDB에 청크 저장 (임베딩 생성 및 컬렉션 생성)
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=str(DB_PATH),
        # collection_metadata={"hnsw:space": "cosine"}
    )

print(f"\n저장된 청크 수: {vector_store._collection.count()}")

기존 ChromaDB가 존재하므로 새로 생성하지 않고 로딩합니다.

저장된 청크 수: 52


### ChromaDB 기본 유사도 계산
| 항목       | Chroma DB 기본값            |
| -------- | ------------------------ |
| 기본 거리 방식 | `l2` (제곱 유클리드 거리)  |
| 의미       | Squared L2 distance      |
| 해석       | 낮을수록 더 유사                |
| 변경 가능 값  | `cosine`, `ip`           |
| 설정 위치    | collection 생성 시 metadata |


# 벡터DB 쿼리 및 LLM 답변 결과 확인

## Naive RAG 한계 확인

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

prompt = ChatPromptTemplate.from_messages([
    ("system", "아래 문서를 참고해서 질문에 답하세요.\n\n[문서]\n{context}"),
    ("human", "{question}")
])

naive_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

In [ ]:
### 표현 방식에 따른 검색 편차
# [질문]
# 1. 2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?
# 2. 내년 AI 모델은 어떤 방향으로 발전할 것으로 보이나요?
# 3. 합성데이터, 추론형 AI, 멀티모달 기술은 앞으로 어떤 역할을 하나요?
# 4. AI 정책 분야에서 안전성과 책임이 왜 중요한 이슈인가요?
# 5. 신뢰할 수 있는 AI를 만들기 위해 어떤 제도적 장치가 필요하다고 하나요?

In [9]:
def inspect_retrieval(query: str, k: int = 5, active_store=vector_store):
    """
    질문에 대해 검색된 문서를 확인한다.

    similarity_search_with_score()의 score는 기본적으로 distance 값이다.
    즉, 낮을수록 질문과 문서가 더 가깝다.
    """

    results = active_store.similarity_search_with_score(query, k=k)

    print("=" * 100)
    print(f"[질문] {query}")
    print(f"[검색 개수 k] {k}")
    print("ChromaDB distance score: 낮을수록 더 유사함")
    print("=" * 100)

    for i, (doc, score) in enumerate(results, start=1):
        page = doc.metadata.get("page", None)
        page_label = page + 1 if isinstance(page, int) else "?"
        chunk_id = doc.metadata.get("chunk_id", "?")
        source = doc.metadata.get("source", "?")

        content = doc.page_content[:500].replace("\n", " ")

        print(f"\n[{i}] score={score:.4f} | page={page_label} | chunk_id={chunk_id}")
        print(f"source={source}")
        print(content)

In [10]:
# 문제 1: 표현 방식에 따른 검색 편차 확인

test_queries = [
    "2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?",
    "내년 AI 모델은 어떤 방향으로 발전할 것으로 보이나요?",
    "합성데이터, 추론형 AI, 멀티모달 기술은 앞으로 어떤 역할을 하나요?",
]

for query in test_queries:
    inspect_retrieval(query, k=3)
    print("\n")

[질문] 2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?
[검색 개수 k] 3
ChromaDB distance score: 낮을수록 더 유사함

[1] score=0.7387 | page=18 | chunk_id=40
source=data\AI@Data_Report_2026_전망_251223(최종).pdf
토픽 분석을 통한 AI 주요 트렌드 및 2026 전망18              ※ IoT 센서와 AI 기술을 통해 물류 및 공급망 전 과정을 자동화하고 최적화4)[ 그림 5 ]주요 산업 분야별 AI 적용 사례4.32026년 AI 기술 전망 2026년 AI 기술은 지능 구조 고도화와 데이터 한계 보완을 중심으로 진화할 전망임Ÿ합성데이터·추론형 AI·멀티모달 기술이 주요 경쟁 축으로 자리 잡으면서, 학습 효율 향상·복합 정보 처리·설명 가능성 강화 등 모델 내부 구조의 질적 개선 흐름이 이어질 것으로 예상됨 기술 고도화는 입력 유형과 데이터 범위를 확장함으로써 AI의 활용 가능성을 넓히고, 복잡한 상황에서의 판단 능력을 강화하는 방향으로 진화할 것으로 전망됨Ÿ고품질 데이터 생성·멀티모달·고급 추론 기술이 결합되며 AI의 상황 이해와 문제 해결 능력이 향상되고, 서비스·산업 전반의 활용도도 확대될 전망임

[2] score=0.7616 | page=19 | chunk_id=42
source=data\AI@Data_Report_2026_전망_251223(최종).pdf
19토픽 분석을 통한 AI 주요 트렌드 및 2026 전망4.42026년 AI 정책 전망 2026년의 AI 정책 환경은 규제를 제약이 아닌 성장을 위한 안전장치로 재정의하며, 글로벌 규제체계와의 정합성을 높이는 방향으로 재편될 것으로 전망됨 고위험 분야의 책임·안전성 확보, 데이터·저작권 정책 명확화, 국제 표준과의 조화(Harmonization) 등이 결합되며 기업의 글로벌 시장 진출을 지원하는 예측 가능한 정책 생태계가 구축될 것으로 예상됨ŸEU AI Act 등 글로벌 규제와의 정합성을 높이

## 실제 Naive RAG 답변 비교

In [11]:
def ask_naive_rag(question: str):
    """
    Naive RAG 체인을 실행하고 답변을 출력한다.
    """
    answer = naive_chain.invoke(question)

    print("=" * 100)
    print(f"[질문]\n{question}")
    print("\n[Naive RAG 답변]")
    print(answer)
    print("=" * 100)

    return answer

In [12]:
rag_questions = [
    "2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?",
    "내년 AI 모델은 어떤 방향으로 발전할 것으로 보이나요?",
    "산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?",
]

for question in rag_questions:
    ask_naive_rag(question)

[질문]
2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?

[Naive RAG 답변]
2026년 AI 기술 전망에서 핵심 변화는 지능 구조의 고도화와 데이터 한계 보완을 중심으로 진화할 것으로 예상됩니다. 주요 변화로는 합성 데이터, 추론형 AI, 멀티모달 기술이 주요 경쟁 축으로 자리 잡으며, 학습 효율 향상, 복합 정보 처리, 설명 가능성 강화 등 모델 내부 구조의 질적 개선이 이루어질 것입니다. 또한, 입력 유형과 데이터 범위를 확장하여 AI의 활용 가능성을 넓히고, 복잡한 상황에서의 판단 능력을 강화하는 방향으로 진화할 것으로 보입니다. 고품질 데이터 생성, 멀티모달, 고급 추론 기술의 결합으로 AI의 상황 이해와 문제 해결 능력이 향상되고, 서비스 및 산업 전반의 활용도도 확대될 전망입니다.
[질문]
내년 AI 모델은 어떤 방향으로 발전할 것으로 보이나요?

[Naive RAG 답변]
내년 AI 모델은 지능 구조의 고도화와 데이터 한계 보완을 중심으로 발전할 것으로 예상됩니다. 합성 데이터, 추론형 AI, 멀티모달 기술이 주요 경쟁 축으로 자리 잡으면서 학습 효율 향상, 복합 정보 처리, 설명 가능성 강화 등 모델 내부 구조의 질적 개선이 이루어질 것입니다. 이러한 기술 고도화는 입력 유형과 데이터 범위를 확장하여 AI의 활용 가능성을 넓히고, 복잡한 상황에서의 판단 능력을 강화하는 방향으로 진화할 것으로 보입니다. 고품질 데이터 생성, 멀티모달, 고급 추론 기술이 결합되어 AI의 상황 이해와 문제 해결 능력이 향상되고, 서비스 및 산업 전반의 활용도도 확대될 전망입니다.
[질문]
산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?

[Naive RAG 답변]
산업, 기술, 정책 분야별 데이터 수집 키워드는 다음과 같습니다:

- **산업**: AI투자, 시장규모, 기업도입, 생성형AI, 펀딩, 스타트업, 산업전환
- **기술**: AI에이전트, 멀티모달, LLM, 온디바이스AI, 오픈소스, 딥시크, 모델개발
- **정책**: AI규

## 검색 문서와 답변 함께 확인하기

In [13]:
def compare_retrieval_and_answer(question: str, k: int = 3):
    """
    검색 결과와 최종 답변을 함께 확인한다.
    Naive RAG에서 검색 품질이 답변 품질에 어떤 영향을 주는지 확인할 수 있다.
    """

    print("\n\n")
    print("#" * 100)
    print("[1] 검색 결과 확인")
    print("#" * 100)
    inspect_retrieval(question, k=k)

    print("\n\n")
    print("#" * 100)
    print("[2] Naive RAG 답변 확인")
    print("#" * 100)
    answer = naive_chain.invoke(question)
    print(answer)

    return answer

In [14]:
compare_retrieval_and_answer(
    "산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?",
    k=3
)




####################################################################################################
[1] 검색 결과 확인
####################################################################################################
[질문] 산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?
[검색 개수 k] 3
ChromaDB distance score: 낮을수록 더 유사함

[1] score=0.7358 | page=10 | chunk_id=20
source=data\AI@Data_Report_2026_전망_251223(최종).pdf
분야별로 설정한 7개 키워드를 기반으로 전체 수집 건수는 282건(분야별 94건)이며, 각 기사·리포트별로 주요 내용 요약본을 정리한 후, 산업·기술·정책별 세부 키워드 분석을 실시함Ÿ해외 매체는 의미 왜곡을 최소화하고 분석 기준을 일관되게 유지하기 위해 한국어로 번역 후 요약하였으며, 이를 통해 분석 절차의 체계성과 비교 가능성을 확보함 분야 사용 키워드AI산업AI투자, 시장규모, 기업도입, 생성형AI, 펀딩, 스타트업, 산업전환AI기술AI에이전트, 멀티모달, LLM, 온디바이스AI, 오픈소스, 딥시크, 모델개발AI정책AI규제, AI기본법, EU AI Act, 거버넌스, 행정명령, 데이터보호, 인프라정책 [ 그림 1 ]분야별 데이터셋 구성 예시

[2] score=0.8341 | page=10 | chunk_id=19
source=data\AI@Data_Report_2026_전망_251223(최종).pdf
토픽 분석을 통한 AI 주요 트렌드 및 2026 전망10Ÿ분야 간 키워드 수를 동일하게 맞춤으로써 비교 과정에서 기준 차이에 따른 왜곡을 최소화함 본 분석의 분야별 핵심 키워드는 국내외 주요 매체에서 반복 등장한 상위 빈도 핵심어 중 산업·기술·정책의 주요 

'산업, 기술, 정책 분야별 데이터 수집 키워드는 다음과 같습니다:\n\n- **산업**: AI투자, 시장규모, 기업도입, 생성형AI, 펀딩, 스타트업, 산업전환\n- **기술**: AI에이전트, 멀티모달, LLM, 온디바이스AI, 오픈소스, 딥시크, 모델개발\n- **정책**: AI규제, AI기본법, EU AI Act, 거버넌스, 행정명령, 데이터보호, 인프라정책'

## 관련 없는 청크도 반환

### 확인 포인트

Naive RAG는 `k=5`로 설정하면 관련성이 낮은 문서가 포함되더라도 무조건 5개의 청크를 반환한다.

이 방식의 문제는 다음과 같다.

1. 관련성이 낮은 문서가 프롬프트에 함께 들어간다.
2. LLM이 중요하지 않은 내용을 함께 참고할 수 있다.
3. 답변이 길어지거나 초점이 흐려질 수 있다.
4. 토큰 사용량이 증가한다.

이 문제는 이후 실습에서 `Reranking`과 `Contextual Compression`으로 개선한다.

In [15]:
### 관련 없는 청크도 반환
# [질문]
# 1. AI 에이전트가 기업 운영 방식에 어떤 변화를 가져올 것으로 전망되나요?
# 2. 보고서에서 말하는 온디바이스 AI 확산은 어떤 의미인가요?
# 3. 토픽모델링에서 단순 출현 빈도보다 중요하게 본 기준은 무엇인가요?
# 4. AI 정책에서 출력물 표시와 데이터 출처 공개는 왜 중요하게 다루어지나요?
# 5. 보고서의 전처리 과정에서 제거한 단어 유형은 무엇인가요?

### ChromaDB의 score 해석

`similarity_search_with_score()`에서 반환되는 score는 일반적인 “유사도 점수”라기보다 거리값에 가깝다.

따라서 기본 설정에서는 다음과 같이 해석한다.

- score가 낮을수록 질문과 문서가 더 가깝다.
- score가 높을수록 질문과 문서의 거리가 멀다.
- ChromaDB의 기본 거리 방식은 L2 distance 계열이다.
  - 유클리드 거리 제곱 = Squared L2 Distance, 벡터 간 거리의 제곱. 기본값
- cosine 기반 거리를 사용하려면 collection 생성 시 별도로 지정해야 한다.

예시:

```python
Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="ai_trend_report_2026_pdf_cosine",
    persist_directory="chroma_db/ai_trend_report_pdf_cosine",
    collection_metadata={"hnsw:space": "cosine"}
)
```

In [16]:
# 문제 2: 관련 없는 청크도 반환
query =  "AI 에이전트가 기업 운영 방식에 어떤 변화를 가져올 것으로 전망되나요?"
inspect_retrieval(query, k=5)

[질문] AI 에이전트가 기업 운영 방식에 어떤 변화를 가져올 것으로 전망되나요?
[검색 개수 k] 5
ChromaDB distance score: 낮을수록 더 유사함

[1] score=0.8724 | page=17 | chunk_id=39
source=data\AI@Data_Report_2026_전망_251223(최종).pdf
기업 내부에서 AI 에이전트를 활용한 문서 처리, 고객 지원, 운영 자동화 등이 증가하며 사람–에이전트–시스템이 혼합된 업무 구조가 일부 영역에서 확산될 가능성이 있음Ÿ기업 간(B2B) 시스템 연동을 통해 제한된 범위에서 AI 간 협업 형태의 자동화 사례도 등장할 것으로 예상되지만, 그 영향은 아직 초기 단계에 머물며 중장기적 구조 변화의 단초를 형성하는 수준으로 평가됨

[2] score=0.9279 | page=13 | chunk_id=29
source=data\AI@Data_Report_2026_전망_251223(최종).pdf
인프라, 센터, 효율, 에이전트: 인프라·에이전트 기반 운영 모델 변화 Ÿ데이터센터·클라우드 기반의 인프라 투자 확대 경향을 반영함Ÿ에이전트 도입 확산은 업무 흐름과 운영 방식이 재구성되는 초기 징후로 볼 수 있음 현시점 AI 산업은 시장 규모 확대, 도입 가속, 인프라 중심 운영 모델화가 동시에 나타나며, AI가 산업의 핵심 인프라로 자리 잡아가는 흐름을 시사함 [ 그림 2 ]시장 규모 확대와 비즈니스 구조 전환 중심 워드 클라우드

[3] score=0.9624 | page=16 | chunk_id=36
source=data\AI@Data_Report_2026_전망_251223(최종).pdf
분야토픽명주요 키워드핵심 이슈AI 산업시장 규모 확대와 비즈니스 구조 전환도입, 확대, 성장세, 확산. 규모, 성장, 글로벌, 분기, 비용, 자금, 인프라, 센터, 효율, 에이전트ŸAI 도입·확산 가속에 따른 산업 구조 변화Ÿ산업 규모 확대 및 글로벌 경쟁 심화Ÿ인프라·에이전트 기반 운영 모델 변화AI 기술기능 고

## 표 기반 검색 한계

In [17]:
### 문제3. 표 기반 검색 한계
# [질문]
# 1. 이 보고서는 AI 트렌드를 분석하기 위해 어떤 분석 절차를 거쳤나요?
# 2. 산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?
# 3. 토픽모델링 전에 제거한 전처리 용어 유형과 예시를 정리해줘.
# 4. AI 산업, AI 기술, AI 정책의 토픽명과 핵심 이슈를 비교해줘.
# 5. 추론형 데이터는 어떤 유형으로 구분되며, 각 유형의 예시는 무엇인가요?

In [18]:
query =  "이 보고서는 AI 트렌드를 분석하기 위해 어떤 분석 과정를 거쳤나요?"
inspect_retrieval(query, k=5)

[질문] 이 보고서는 AI 트렌드를 분석하기 위해 어떤 분석 과정를 거쳤나요?
[검색 개수 k] 5
ChromaDB distance score: 낮을수록 더 유사함

[1] score=0.7382 | page=11 | chunk_id=21
source=data\AI@Data_Report_2026_전망_251223(최종).pdf
11토픽 분석을 통한 AI 주요 트렌드 및 2026 전망2.2연구 방법

[2] score=0.7768 | page=8 | chunk_id=14
source=data\AI@Data_Report_2026_전망_251223(최종).pdf
토픽 분석을 통한 AI 주요 트렌드 및 2026 전망81.3분석 한계 및 유의사항 본 분석은 2025년 1월부터 11월까지 국내·해외 주요 언론 및 공공·산업 데이터를 중심으로 수행되었으며, 기사·보고서·정책 자료의 주기적 스크리닝과 키워드 정제에 기반하고 있음Ÿ해외 비언론 데이터, 최신 산업·기술 사례, 기업 내부 자료 등은 자료 접근성과 범위 한계로 인해 깊이 있게 반영되지 못하였음Ÿ특정 기간(11개월)의 담론 흐름에 기반하므로, 빠르게 변화하는 AI 환경의 시간적 민감성을 전부 포착하기에는 어려움이 존재함

[3] score=0.7928 | page=21 | chunk_id=47
source=data\AI@Data_Report_2026_전망_251223(최종).pdf
21토픽 분석을 통한 AI 주요 트렌드 및 2026 전망

[4] score=0.8520 | page=8 | chunk_id=16
source=data\AI@Data_Report_2026_전망_251223(최종).pdf
기사·보고서 기반 분석은 실제 AI 트렌드의 전체 맥락과 세부 이슈를 완전히 포괄하기 어려우므로, 향후 연구 및 정책 제안에서는 보다 다양한 데이터 소스 확충과 다층적 분석, 심층 사례 검토가 필요함Ÿ언론의 이슈 편향, 국가별 리스크 인식 차이, 기업 홍보성 보도 등으로 인해 담론 기반 분석은 구조적 편향(Structura

In [19]:
query =  "산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?"
inspect_retrieval(query, k=5)

[질문] 산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?
[검색 개수 k] 5
ChromaDB distance score: 낮을수록 더 유사함

[1] score=0.7359 | page=10 | chunk_id=20
source=data\AI@Data_Report_2026_전망_251223(최종).pdf
분야별로 설정한 7개 키워드를 기반으로 전체 수집 건수는 282건(분야별 94건)이며, 각 기사·리포트별로 주요 내용 요약본을 정리한 후, 산업·기술·정책별 세부 키워드 분석을 실시함Ÿ해외 매체는 의미 왜곡을 최소화하고 분석 기준을 일관되게 유지하기 위해 한국어로 번역 후 요약하였으며, 이를 통해 분석 절차의 체계성과 비교 가능성을 확보함 분야 사용 키워드AI산업AI투자, 시장규모, 기업도입, 생성형AI, 펀딩, 스타트업, 산업전환AI기술AI에이전트, 멀티모달, LLM, 온디바이스AI, 오픈소스, 딥시크, 모델개발AI정책AI규제, AI기본법, EU AI Act, 거버넌스, 행정명령, 데이터보호, 인프라정책 [ 그림 1 ]분야별 데이터셋 구성 예시

[2] score=0.8342 | page=10 | chunk_id=19
source=data\AI@Data_Report_2026_전망_251223(최종).pdf
토픽 분석을 통한 AI 주요 트렌드 및 2026 전망10Ÿ분야 간 키워드 수를 동일하게 맞춤으로써 비교 과정에서 기준 차이에 따른 왜곡을 최소화함 본 분석의 분야별 핵심 키워드는 국내외 주요 매체에서 반복 등장한 상위 빈도 핵심어 중 산업·기술·정책의 주요 이슈와 변화 방향을 가장 정확히 대표하는 단어들을 선별하여 구성Ÿ핵심 키워드는 산업·기술·정책 각 분야에서 반복적으로 등장한 상위 빈도 핵심어 중 대표성이 가장 높은 단어를 기반으로 선정함Ÿ이는 각 키워드가 분야별 담론 구조와 주요 이슈·변화 방향(시장 확산, 기술 고도화, 규제 강화 등)을 가장 명확하게 설명하는 지표 역할을 하기 때문임< 표 2 >분야별 데이터 수집 키워드

[3] s

In [20]:
compare_retrieval_and_answer(
    "산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?",
    k=5
)




####################################################################################################
[1] 검색 결과 확인
####################################################################################################
[질문] 산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?
[검색 개수 k] 5
ChromaDB distance score: 낮을수록 더 유사함

[1] score=0.7358 | page=10 | chunk_id=20
source=data\AI@Data_Report_2026_전망_251223(최종).pdf
분야별로 설정한 7개 키워드를 기반으로 전체 수집 건수는 282건(분야별 94건)이며, 각 기사·리포트별로 주요 내용 요약본을 정리한 후, 산업·기술·정책별 세부 키워드 분석을 실시함Ÿ해외 매체는 의미 왜곡을 최소화하고 분석 기준을 일관되게 유지하기 위해 한국어로 번역 후 요약하였으며, 이를 통해 분석 절차의 체계성과 비교 가능성을 확보함 분야 사용 키워드AI산업AI투자, 시장규모, 기업도입, 생성형AI, 펀딩, 스타트업, 산업전환AI기술AI에이전트, 멀티모달, LLM, 온디바이스AI, 오픈소스, 딥시크, 모델개발AI정책AI규제, AI기본법, EU AI Act, 거버넌스, 행정명령, 데이터보호, 인프라정책 [ 그림 1 ]분야별 데이터셋 구성 예시

[2] score=0.8341 | page=10 | chunk_id=19
source=data\AI@Data_Report_2026_전망_251223(최종).pdf
토픽 분석을 통한 AI 주요 트렌드 및 2026 전망10Ÿ분야 간 키워드 수를 동일하게 맞춤으로써 비교 과정에서 기준 차이에 따른 왜곡을 최소화함 본 분석의 분야별 핵심 키워드는 국내외 주요 매체에서 반복 등장한 상위 빈도 핵심어 중 산업·기술·정책의 주요 

'산업, 기술, 정책 분야별 데이터 수집 키워드는 다음과 같습니다:\n\n- **산업**: AI투자, 시장규모, 기업도입, 생성형AI, 펀딩, 스타트업, 산업전환\n- **기술**: AI에이전트, 멀티모달, LLM, 온디바이스AI, 오픈소스, 딥시크, 모델개발\n- **정책**: AI규제, AI기본법, EU AI Act, 거버넌스, 행정명령, 데이터보호, 인프라정책'

## PDF 직접 로딩 방식의 한계

PDF 문서를 직접 로딩하면 표, 목차, 본문 구조가 하나의 긴 문자열처럼 추출될 수 있다.  
이 경우 사람에게는 표처럼 보이는 정보도, 검색 모델에게는 구조가 흐트러진 텍스트로 인식될 수 있다.

예를 들어 다음과 같은 문제가 발생할 수 있다.

1. 표의 행과 열 관계가 약해진다.
2. 항목 간 구분이 불명확해진다.
3. 키워드 목록이 한 줄로 붙어서 검색된다.
4. 답변 생성 시 분야별 구분이 흐려질 수 있다.

따라서 다음 실습에서는 PDF를 Markdown으로 변환하여 문서 구조를 더 잘 보존하는 방식을 사용한다.

## 정리

이번 실습에서는 PDF 문서를 직접 로딩하여 Naive RAG를 구현하고, 그 한계를 확인했다.

### 확인한 내용

1. `PyPDFLoader`는 PDF를 페이지 단위 Document로 로딩한다.
2. `RecursiveCharacterTextSplitter`는 문서를 청크 단위로 분할한다.
3. ChromaDB는 문서 청크와 임베딩 벡터를 로컬에 저장할 수 있다.
4. Naive RAG는 단일 질문을 그대로 검색에 사용한다.
5. 질문 표현이 달라지면 검색 결과가 달라질 수 있다.
6. 고정된 `k`개 검색 방식은 관련성이 낮은 청크도 함께 반환할 수 있다.
7. PDF 직접 로딩은 표, 목차, 문단 구조가 흐트러질 수 있다.